<a href="https://colab.research.google.com/github/sahmedshereen-prog/recommendation-System/blob/main/Task_3_Uneeq.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
"""
Recommendation System - Movie Recommender
============================================
Task: Build a recommendation system for a movie platform.

Dataset: MovieLens (latest-small)
Link: https://www.kaggle.com/datasets/grouplens/movielens-latest-small
Files used (place both next to this script):
- movies.csv   (movieId, title, genres)
- ratings.csv  (userId, movieId, rating, timestamp)

This script implements TWO recommendation approaches:

1. COLLABORATIVE FILTERING (item-based)
   Recommends movies based on how OTHER USERS with similar taste rated them.
   Uses cosine similarity between movies based on the user-rating matrix.

2. CONTENT-BASED FILTERING
   Recommends movies similar in GENRE to a movie the user liked.
   Uses TF-IDF on genres + cosine similarity.

Both are evaluated:
- Collaborative filtering: RMSE on a held-out test set of ratings
- Content-based: qualitative check (top-N similar movies for a sample title)
"""

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import mean_squared_error

movies = pd.read_csv("movies.csv")
ratings = pd.read_csv("ratings.csv")

print("Movies shape:", movies.shape)
print("Ratings shape:", ratings.shape)
print(movies.head())
print(ratings.head())

train_ratings, test_ratings = train_test_split(
    ratings, test_size=0.2, random_state=42
)

user_item_matrix = train_ratings.pivot_table(
    index="userId", columns="movieId", values="rating"
).fillna(0)

item_similarity = cosine_similarity(user_item_matrix.T)
item_similarity_df = pd.DataFrame(
    item_similarity,
    index=user_item_matrix.columns,
    columns=user_item_matrix.columns,
)

def predict_rating(user_id, movie_id, k=20):
    """Predict a user's rating for a movie using a weighted average of
    ratings they gave to the k most similar movies."""
    if movie_id not in item_similarity_df.columns or user_id not in user_item_matrix.index:
        return train_ratings["rating"].mean()

    user_ratings = user_item_matrix.loc[user_id]
    rated_movies = user_ratings[user_ratings > 0]

    if rated_movies.empty:
        return train_ratings["rating"].mean()

    sims = item_similarity_df.loc[movie_id, rated_movies.index]
    top_k = sims.sort_values(ascending=False).head(k)
    top_k = top_k[top_k > 0]

    if top_k.empty:
        return train_ratings["rating"].mean()

    weighted_sum = (top_k * rated_movies[top_k.index]).sum()
    sim_sum = top_k.sum()
    return weighted_sum / sim_sum if sim_sum > 0 else train_ratings["rating"].mean()

def recommend_movies_for_user(user_id, top_n=10):
    """Recommend top-N movies for a user that they haven't rated yet."""
    if user_id not in user_item_matrix.index:
        return pd.DataFrame(columns=["title", "predicted_rating"])

    already_rated = set(user_item_matrix.loc[user_id][user_item_matrix.loc[user_id] > 0].index)
    candidates = [m for m in user_item_matrix.columns if m not in already_rated]

    predictions = [(m, predict_rating(user_id, m)) for m in candidates]
    predictions.sort(key=lambda x: x[1], reverse=True)
    top_movies = predictions[:top_n]

    result = pd.DataFrame(top_movies, columns=["movieId", "predicted_rating"])
    result = result.merge(movies[["movieId", "title"]], on="movieId")
    return result[["title", "predicted_rating"]]

print("\nEvaluating collaborative filtering (this may take a minute)...")
sample_test = test_ratings.sample(min(500, len(test_ratings)), random_state=42)

y_true = []
y_pred = []
for _, row in sample_test.iterrows():
    pred = predict_rating(row["userId"], row["movieId"])
    y_true.append(row["rating"])
    y_pred.append(pred)

rmse = np.sqrt(mean_squared_error(y_true, y_pred))
print(f"Collaborative Filtering RMSE (sampled test set): {rmse:.4f}")

sample_user = ratings["userId"].iloc[0]
print(f"\nTop 10 collaborative-filtering recommendations for user {sample_user}:")
print(recommend_movies_for_user(sample_user, top_n=10))

movies["genres"] = movies["genres"].fillna("").str.replace("|", " ", regex=False)

tfidf = TfidfVectorizer()
genre_matrix = tfidf.fit_transform(movies["genres"])

genre_similarity = cosine_similarity(genre_matrix)
title_to_index = pd.Series(movies.index, index=movies["title"]).drop_duplicates()

def recommend_similar_movies(title, top_n=10):
    """Recommend movies similar in genre to the given movie title."""
    if title not in title_to_index:
        return f"'{title}' not found in the dataset."

    idx = title_to_index[title]
    sim_scores = list(enumerate(genre_similarity[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n + 1]  # skip itself

    movie_indices = [i[0] for i in sim_scores]
    return movies.iloc[movie_indices][["title", "genres"]]

sample_title = "Toy Story (1995)"
print(f"\nTop 10 content-based recommendations similar to '{sample_title}':")
print(recommend_similar_movies(sample_title, top_n=10))

print("\nDone.")

Movies shape: (9742, 3)
Ratings shape: (100836, 4)
   movieId                               title  \
0        1                    Toy Story (1995)   
1        2                      Jumanji (1995)   
2        3             Grumpier Old Men (1995)   
3        4            Waiting to Exhale (1995)   
4        5  Father of the Bride Part II (1995)   

                                        genres  
0  Adventure|Animation|Children|Comedy|Fantasy  
1                   Adventure|Children|Fantasy  
2                               Comedy|Romance  
3                         Comedy|Drama|Romance  
4                                       Comedy  
   userId  movieId  rating  timestamp
0       1        1     4.0  964982703
1       1        3     4.0  964981247
2       1        6     4.0  964982224
3       1       47     5.0  964983815
4       1       50     5.0  964982931

Evaluating collaborative filtering (this may take a minute)...
Collaborative Filtering RMSE (sampled test set): 0.8785

Top 1